# 從零理解 RAG：Cohere 雲端檢索與本地 Phi-3 問答

> **使用的工具**：Cohere API（嵌入、重新排名、生成）· FAISS（向量搜尋）· BM25（關鍵字搜尋）· 本地 `Phi-3-mini`（GGUF）· `gte-small`（本地嵌入模型）· LangChain

---

## 這堂課你會學到什麼

1. **RAG 是什麼、為什麼需要它**：模型不知道你的文件，直接問會亂猜；先「找資料」再「回答」才可靠
2. **文字切塊（chunking）**：清理 → 切句 → 用「滑動視窗」把句子組成段落
3. **密集檢索（Dense Retrieval）**：用嵌入向量 + FAISS，依「意思」找資料
4. **關鍵字檢索 + 重新排名**：BM25 先粗篩，再請 Cohere Rerank 精排
5. **兩種生成方式**：Cohere 的 grounded generation（雲端）與本地 RAG（Phi-3 + gte-small + FAISS）
6. **三種檢索各自的強弱**，以及這份程式裡幾個值得注意的陷阱

## 0. 環境準備

```bash
pip install cohere numpy pandas tqdm faiss-cpu rank-bm25 scikit-learn python-dotenv
pip install langchain-community langchain-huggingface langchain-core langchain-classic llama-cpp-python
```

| 準備項目 | 說明 |
|---|---|
| **Python 3.12 以上** | 程式裡有 `f"... {x.replace("\n", " ")}"` 這種「f-string 大括號裡又用同一種引號」的寫法，這在 Python 3.12 才支援 |
| **Cohere 金鑰** | 在程式同一資料夾建立 `.env`，內容 `COHERE_API_KEY=你的金鑰`；並把 `.env` 加進 `.gitignore` |
| **Phi-3 模型檔** | `Phi-3-mini-4k-instruct-q4.gguf`，放在程式的上一層資料夾（`../`），或改 `LOCAL_LLM_MODEL` 的路徑 |
| **gte-small** | 第一次執行會自動從 Hugging Face 下載，需要網路與硬碟空間 |

## 1. 先看整體：RAG 是什麼？

### 為什麼需要 RAG？

直接問 LLM「Interstellar 賺了多少錢」，它只能靠訓練時記住的內容回答，可能記錯、也可能一本正經地亂說。**RAG（Retrieval-Augmented Generation，檢索增強生成）** 的做法是：

```
問題 ──► ① 檢索：從你的文件裡找出最相關的幾段 ──► ② 生成：把「問題 + 找到的段落」一起交給 LLM 回答
```

LLM 這時是「看著資料回答」，而不是「憑記憶回答」。

### 這份程式有三條路線

```
                        ┌─ (A) 密集檢索 dense_search ──► Cohere 生成 ask      （雲端：嵌入 + 生成都用 Cohere API）
 文章 ─► 清理 ─► 切塊 ──┤
                        ├─ (B) BM25 關鍵字檢索 ──► Cohere Rerank 重排       （只印出排名，沒有接生成）
                        │
                        └─ (C) 本地 RAG：gte-small + FAISS + Phi-3          （本地：嵌入、檢索、生成都在自己電腦）
```

| 路線 | 怎麼找資料 | 誰負責生成 | 需要網路嗎 |
|---|---|---|---|
| A. 密集檢索 + `ask` | 嵌入向量，找「意思相近」的段落 | Cohere `chat` | 要（Cohere API） |
| B. BM25 + Rerank | 先比對「詞有沒有出現」，再由 Rerank 模型精排 | 無（只印出排名） | 要（Rerank 用 Cohere API） |
| C. 本地 RAG | 本地嵌入模型 + LangChain 的 FAISS | 本地 Phi-3 | 第一次下載模型要，之後不用 |

### 這堂課出現的名詞

| 名詞 | 意思 |
|---|---|
| Chunk（塊） | 切好的一小段文字，是檢索的最小單位 |
| Embedding（嵌入向量） | 把一段文字變成一串數字，意思越接近，數字越接近 |
| Dense retrieval（密集檢索） | 用嵌入向量的距離找資料 |
| BM25 | 傳統的關鍵字搜尋演算法，看查詢詞在段落裡出現的情況打分數 |
| Rerank（重新排名） | 用更精細的模型，把初步找到的候選重新排順序 |
| Grounded generation | 生成時直接附上資料，要求模型依據這些資料回答 |
| FAISS | Meta 開發的向量搜尋函式庫 |

## 2. 匯入套件

| 匯入 | 用途 |
|---|---|
| `cohere` | 呼叫 Cohere API（嵌入、重新排名、生成） |
| `numpy` / `pandas` | 處理向量（陣列）/ 把搜尋結果整理成表格 |
| `tqdm` | 顯示進度條 |
| `faiss` | 向量搜尋引擎本身（密集檢索用） |
| `rank_bm25.BM25Okapi` | BM25 關鍵字搜尋 |
| `string` / `sklearn ... _stop_words` | 斷詞時去掉標點符號與英文停用字（the、a、is…） |
| `LlamaCpp` | 在本機執行 GGUF 格式的 LLM（Phi-3） |
| `HuggingFaceEmbeddings` | 在本機執行嵌入模型（gte-small） |
| `FAISS`（LangChain） | 把 FAISS 包成 LangChain 能直接用的向量資料庫 |
| `PromptTemplate` | 提示詞模板 |
| `RetrievalQA` | 把「檢索 + 生成」串成一條鏈；這是舊式的鏈，所以放在 `langchain_classic` |
| `re` / `os` / `load_dotenv` | 正規表示式 / 讀環境變數 / 讀 `.env` |

In [1]:
#'thenlper/gte-small'   文本嵌入模型
#../Phi-3-mini-4k-instruct-q4.gguf  LLM  生成模型
#BM25Okapi 關鍵字搜尋

import cohere
import numpy as np
import pandas as pd
from tqdm import tqdm             #顯示動態進度條的第三方庫
import faiss                      #Meta 開發的向量搜尋函式庫本身 自己算 embedding、轉 numpy、管理索引
from rank_bm25 import BM25Okapi   #關鍵字搜尋函數
import string
from sklearn.feature_extraction import _stop_words
from langchain_community.llms import LlamaCpp   #Meta開發的LLM 能在Python 環境中，用簡單的幾行程式碼載入並運行 GGUF 格式的大語言模型。
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS    #import faiss 是底層的向量搜尋引擎，FAISS（LangChain）是把它包裝好的高階工具。
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA
import re
import os
from dotenv import load_dotenv

/tmp/ipykernel_11387/1669943599.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import LlamaCpp   #Meta開發的LLM 能在Python 環境中，用簡單的幾行程式碼載入並運行 GGUF 格式的大語言模型。


## 3. 建立 Cohere 客戶端

- `EMBED_MODEL`：嵌入模型名稱，單獨放成變數方便更換。
- `load_dotenv()` 把 `.env` 的內容讀進環境變數，`os.getenv("COHERE_API_KEY")` 再取出金鑰。**金鑰不要寫在程式碼裡。**
- `cohere.Client(api_key)` 建立「可以跟 Cohere API 溝通的客戶端物件」，之後所有 API 呼叫都透過它。

In [2]:
EMBED_MODEL = "embed-v4.0"

embed_model = EMBED_MODEL

load_dotenv()
api_key = os.getenv("COHERE_API_KEY")
cohere_client = cohere.Client(api_key)

## 4. 準備文本：清理 → 切句 → 切塊

檢索的單位不是整篇文章，而是「塊」。塊太大，一塊裡混了很多話題；塊太小，單獨一塊看不懂。這份程式的做法是三步：

1. **清理**：`re.sub(r'\[[^\]]*\]', '', ...)` 把維基百科的引用標記（`[b]`、`[1]`、`[9]`）刪掉。
2. **切句**：`re.split(r'(?<=[.!?])\s+', ...)` 在「句號 / 驚嘆號 / 問號 **後面接空白**」的地方切開。`$774.7` 的小數點後面沒有空白，所以不會被切斷。
3. **滑動視窗**：每 3 句合成一塊，每次往後移 1 句，所以**相鄰的塊會重疊**。

```
句子：   s0   s1   s2   s3   s4   s5 ...
第 0 塊：[s0   s1   s2]
第 1 塊：     [s1   s2   s3]
第 2 塊：          [s2   s3   s4]
```

重疊的好處：一個意思如果剛好橫跨兩句，至少有一塊會完整包含它。塊數 = 句子數 − window + 1。

`TEXTS` 是三段維基百科的文字。`texts = TEXTS` 之後，`texts` 這個變數會先後變成「清理後的字串」→「塊的 list」。

In [3]:
TEXTS = """
Interstellar is a 2014 epic[b] science fiction drama film[1][9] directed by Christopher Nolan, who co-wrote the screenplay with his brother, Jonathan Nolan. It features an ensemble cast led by Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, and Michael Caine. Set in a dystopian future where Earth is suffering from catastrophic blight and famine, the film follows a group of astronauts who travel through space in search of a new home for humanity.

The film began development in 2006 from a treatment by producer Lynda Obst and theoretical physicist Kip Thorne, and was originally set to be directed by Steven Spielberg. Thorne served as an executive producer and scientific consultant on the film, and wrote the tie-in book The Science of Interstellar. It was Obst's final film as producer before her death. Cinematographer Hoyte van Hoytema shot it on 35 mm film in the Panavision anamorphic format and IMAX 70 mm. Filming began in late 2013 and took place in Alberta, Klaustur, and Los Angeles. Interstellar uses extensive practical and miniature effects, and the company DNEG created additional visual effects.

Interstellar premiered at the TCL Chinese Theatre on October 26, 2014, and was released in theaters in the United States on November 5, and in the United Kingdom on November 7, with Paramount Pictures distributing in the United States and Warner Bros. Pictures distributing in international markets. In the United States, it was first released on film stock, expanding to venues using digital projectors. It was a commercial success, grossing $681 million worldwide during its initial theatrical run, and $774.7 million worldwide with subsequent releases, making it the 10th-highest-grossing film of 2014. The film received generally positive reviews from critics. Among its various accolades, Interstellar was nominated for five awards at the 87th Academy Awards, winning for Best Visual Effects.
"""

texts = TEXTS

# print(texts)
texts = re.sub(r'\[[^\]]*\]', '', texts)
sentences = re.split(r'(?<=[.!?])\s+', texts.strip())
sentences = [s.strip() for s in sentences if s.strip()]
# print(sentences)
# 3. 每 3 句合成一塊，每次往後移 1 句（相鄰塊會重疊）
window = 3
texts = [
    " ".join(sentences[i:i + window])
    for i in range(max(1, len(sentences) - window + 1))
]

## 5. 把每一塊變成嵌入向量

**嵌入（embedding）** 是把一段文字變成一串數字（向量）。意思越接近的文字，向量在空間裡的位置越接近。這樣「找意思相近的段落」就變成了「找距離最近的向量」。

```
"How much did the film earn?"  ──►  [0.12, -0.53, 0.08, ...]   ← 一串數字
```

重點參數：

| 參數 | 意思 |
|---|---|
| `texts` | 要轉換的文字 list，一次可以送很多塊 |
| `model` | 嵌入模型 |
| `input_type='search_document'` | 告訴模型：「這些是**被搜尋的文件**」。等一下搜尋時，問題會改用 `search_query`，Cohere 的嵌入模型會依用途分別處理 |

這一步會**呼叫 Cohere API**，回傳的 `.embeddings` 是 list；`np.array(...)` 把它變成 numpy 陣列，形狀是「塊數 × 向量維度」。

In [4]:
texts_embeds = cohere_client.embed(
    texts = texts,
    model = embed_model,
    input_type = 'search_document',
).embeddings

texts_embeds = np.array(texts_embeds)

## 6. 建立 FAISS 索引

向量有了，還需要一個「可以快速找最近鄰居」的資料結構，這就是**索引（index）**。

- `faiss.IndexFlatL2(d)`：
  - `d` 是向量維度，所以用 `texts_embeds.shape[1]` 取得。
  - `Flat` 的意思是「不壓縮、不近似，逐一計算距離」，結果精確，資料少的時候很合適。
  - `L2` 是距離的種類（歐氏距離）：**距離越小，越相似。**
- `np.float32(...)`：FAISS 使用 32 位元浮點數，所以先轉型。
- `index.add(...)`：把所有塊的向量放進索引。

In [5]:
#這個函式負責將 embeddings 加入到 FAISS 索引（index）中。
# 一開始是先將 embeddings 轉換為 numpy 的陣列，然後用 texts_embeds.shape[1] 取得每個 embedding 的維度
# 放進 IndexFlatL2 裡
# print(texts_embeds.shape)
index = faiss.IndexFlatL2(texts_embeds.shape[1])
# print(index.is_trained)
index.add(np.float32(texts_embeds))

## 7. 密集檢索（Dense Retrieval）：`dense_search`

流程：

```
問題 ──► Cohere 轉成嵌入向量（input_type="search_query"）──► index.search 找最近的 k 塊 ──► 整理成表格
```

細節：

- 這裡的 `input_type` 是 `"search_query"`（前面建索引時是 `"search_document"`），因為這次要轉換的是「問題」。
- `np.float32([query_embed])`：外面多包一層 `[]`，因為 FAISS 的 `search` 一次可以查多個問題，要給「二維陣列」。所以回傳的 `distances`、`similar_item_ids` 也是二維，要用 `[0]` 取出第一個（也是唯一一個）問題的結果。
- `texts_np[similar_item_ids[0]]`：numpy 允許用「編號 list」一次取出多個元素。
- 最後用 `pd.DataFrame` 做成兩欄的表格：`texts`（找到的塊）與 `distances`（距離，**越小越相似**）。

### 為什麼這裡用函式 `def`？

因為後面的 `ask` 也會用到它，而且你可以用不同問題反覆呼叫。函式裡讀取的 `cohere_client`、`embed_model`、`index`、`texts` 都是前面已經建立好的變數。

In [6]:
def dense_search(query, number_of_results = 5):
    query_embed = cohere_client.embed(                  #獲取query的嵌入向量
    texts = [query],
    model = embed_model,
    input_type = "search_query",
    ).embeddings[0]
    # print(query_embed)
    # print(type(query_embed))

    distances, similar_item_ids = index.search(        #檢索最近的鄰居
        np.float32([query_embed]),
        k = number_of_results,
    )
    # print(distances)
    # print(similar_item_ids)

    texts_np = np.array(texts)                         #將文本轉換為numpy以便於索引
    results = pd.DataFrame( data =
                            {
                                "texts" : texts_np[similar_item_ids[0]],
                                "distances" : distances[0]
                            }
                        )
    return results

In [7]:
query = "how precise was the science"
query_2 = "How much income did Interstellar generate?"

resposne = dense_search(query=query)
print(resposne)

                                               texts  distances
0  Thorne served as an executive producer and sci...   1.520363
1  The film began development in 2006 from a trea...   1.535106
2  Set in a dystopian future where Earth is suffe...   1.536685
3  Interstellar is a 2014 epic science fiction dr...   1.624009
4  It features an ensemble cast led by Matthew Mc...   1.643721


## 8. 關鍵字檢索（BM25）：先把文章變成「乾淨的 token 清單」

BM25 不看意思，**只看詞**：查詢詞在某一塊出現得越多、而且這個詞在全部塊裡越少見，那一塊的分數就越高。所以文章要先切成詞（token）。

`bm25_tokenizer` 做三件事：

1. `text.lower().split()`：全部轉小寫，用空白切開。
2. `token.strip(string.punctuation)`：把詞**前後**的標點符號拿掉（`"worldwide,"` → `"worldwide"`，`"$681"` → `"681"`）。
3. 去掉長度為 0 的詞，以及英文停用字（the、a、an、is…，出現太多、沒有區分力）。

`texts_tokenize` 的部分，則把每一塊都用這個函式處理，得到「list 裡面還是 list」的 `tokenized_corpus`。`tqdm` 讓你看到進度條。

> `bm25_tokenizer` 要先定義，下面的迴圈才能呼叫它。（原本在類別裡它是 `@staticmethod`：不需要用到物件任何資料的方法。）

In [8]:
def bm25_tokenizer(text):
    tokenized_doc = []
    for token in text.lower().split():
        token = token.strip(string.punctuation)  #punctuation(標點符號) 把token前後的標點符號拿掉
        if len(token) > 0 and token not in _stop_words.ENGLISH_STOP_WORDS:   #token 不能是英文 stop word(the a an is...)
            tokenized_doc.append(token)
    return tokenized_doc

In [9]:
tokenized_corpus = []
for passage in tqdm(texts):
    tokenized_doc = bm25_tokenizer(passage)
    tokenized_corpus.append(tokenized_doc)

100%|██████████| 13/13 [00:00<00:00, 52529.82it/s]


In [10]:
# 🧪 教學小實驗（非原程式）：看 bm25_tokenizer 把一句話變成什麼
print(bm25_tokenizer("It grossed $681 million worldwide, and $774.7 million with subsequent releases."))

['grossed', '681', 'million', 'worldwide', '774.7', 'million', 'subsequent', 'releases']


## 9. 關鍵字檢索 + 重新排名：`keyword_and_rerank_search`

這是典型的**兩階段檢索**：

```
全部 13 塊 ──► 第一階段 BM25：便宜、快，先選出 num_candidates 個候選
          ──► 第二階段 Rerank：精準、較貴（要呼叫 API），把候選重新排序，取前 top_k 名
```

第一階段的程式細節：

| 程式 | 做了什麼 |
|---|---|
| `bm25.get_scores(...)` | 對**每一塊**算出 BM25 分數 |
| `np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]` | 「部分排序」：只保證最大的 `num_candidates` 個排在最後面，不用把全部排好。回傳的是**位置（索引）** |
| `sorted(..., key=lambda x: x['score'], reverse=True)` | 把候選依分數由大到小排好 |

第二階段：`cohere_client.rerank(...)` 會把「問題」和「每個候選段落」逐一配對、打分數，回傳 `relevance_score` 最高的 `top_k` 個。

### 為什麼這裡也用函式 `def`？

它的輸入是 `query`，你可以換不同問題反覆呼叫。注意 `num_candidates` 與 `top_k` 是函式的**預設參數**，不傳就用 10 與 5。

In [11]:
def keyword_and_rerank_search(query, num_candidates = 10, top_k = 5):
    bm25 = BM25Okapi(tokenized_corpus)
    print("Input question:", query)

    bm25_scores = bm25.get_scores(bm25_tokenizer(query))

    #argpartition()部份排序 並回傳索引 kth（正 從其面數, 負 從後面數）指定的第 kth 個元素進行分割。
    # 分割後，kth 位置左側的元素都 <= 該位置的元素，右側的元素都 >= 該位置的元素
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]   #[-num_candidates:]大的都被排在後面 從後面取大的那幾個
    bm25_hits = [{'corpus_id' : idx, 'score' : bm25_scores[idx]} for idx in top_n]

    #lambda x: x['score'] 等價於 def get_score(x): return x["score"], x 是 list 裡「當下正在被拿來比較的那一筆 dictionary」
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse = True)

    print(f"Top 3 lexical search (BM25) hits")
    for hit in bm25_hits[0: top_k]:
        print(
            f"{ hit['score'] : .3f} \t {texts[hit['corpus_id']].replace("\n", " ") }"   #replace是把換行符號 \n 換成空格
        )

    #=======================ReRank==============================================================
    print(f"\n Top3 hits by rerank-API({len(bm25_hits)} BM25 hits re-ranked)")
    docs = [texts[hit['corpus_id']] for hit in bm25_hits]
    rerank_result = cohere_client.rerank(query = query, documents = docs, top_n = top_k, return_documents = True)   #逐步匹配query跟每一個初步篩選後的文本
    # print(rerank_result.results)
    for hit in rerank_result.results:
        # print(hit)
        print(f"{hit.relevance_score : .3f} \t {hit.document.text.replace("\n", " ")}")

In [12]:
keyword_and_rerank_search(query=query)

Input question: how precise was the science
Top 3 lexical search (BM25) hits
 0.821 	 Thorne served as an executive producer and scientific consultant on the film, and wrote the tie-in book The Science of Interstellar. It was Obst's final film as producer before her death. Cinematographer Hoyte van Hoytema shot it on 35 mm film in the Panavision anamorphic format and IMAX 70 mm.
 0.791 	 The film began development in 2006 from a treatment by producer Lynda Obst and theoretical physicist Kip Thorne, and was originally set to be directed by Steven Spielberg. Thorne served as an executive producer and scientific consultant on the film, and wrote the tie-in book The Science of Interstellar. It was Obst's final film as producer before her death.
 0.682 	 Set in a dystopian future where Earth is suffering from catastrophic blight and famine, the film follows a group of astronauts who travel through space in search of a new home for humanity. The film began development in 2006 from a treatmen

## 10. 用 Cohere 做 grounded generation：`grounded_generation` 與 `ask`

**Grounded generation（基於事實的生成）**：把「問題」與「找到的資料」一起交給模型，要求它依據資料回答。

- `docs_dict = [{'text': text} for text in results['texts']]`：這裡把文件整理成「每個元素是一個 dict、內容放在 `text` 欄位」的 list，交給 `documents` 參數。`results['texts']` 就是前面 `dense_search` 的搜尋結果那一欄。
- `cohere_client.chat(message=query, documents=docs_dict)`：回傳的物件除了 `.text`（答案），也帶有引用資訊（`citations`），可以知道答案引用了哪些文件。
- `ask` 把兩步串起來：**搜尋 → 生成 → 取出答案文字**。

整條路線 A 就是：`問題 → dense_search → grounded_generation → .text`。

In [ ]:
def grounded_generation(query, results):
    docs_dict = [{'text' : text} for text in results['texts']]
    # print(docs_dict)
    response = cohere_client.chat(
        message=query,
        documents=docs_dict,
    )
    return response

In [14]:
def ask(query):
    results = dense_search(query)
    grounded_response = grounded_generation(query, results)
    return grounded_response.text

In [15]:
grounded_response_2 = ask(query_2)
print(grounded_response_2)

[{'text': 'It was a commercial success, grossing $681 million worldwide during its initial theatrical run, and $774.7 million worldwide with subsequent releases, making it the 10th-highest-grossing film of 2014. The film received generally positive reviews from critics. Among its various accolades, Interstellar was nominated for five awards at the 87th Academy Awards, winning for Best Visual Effects.'}, {'text': 'Interstellar is a 2014 epic science fiction drama film directed by Christopher Nolan, who co-wrote the screenplay with his brother, Jonathan Nolan. It features an ensemble cast led by Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, and Michael Caine. Set in a dystopian future where Earth is suffering from catastrophic blight and famine, the film follows a group of astronauts who travel through space in search of a new home for humanity.'}, {'text': 'Filming began in late 2013 and took place in Alberta, Klaustur, and Los Angeles. Interstellar us

## 11. 本地 RAG：Phi-3 + gte-small + FAISS + RetrievalQA

路線 C 的所有東西都在自己的電腦上執行。先看「零件」：

| 零件 | 程式 | 做什麼 |
|---|---|---|
| 生成模型 | `LlamaCpp(...)` | 載入 Phi-3 的 GGUF 檔 |
| 嵌入模型 | `HuggingFaceEmbeddings(model_name='thenlper/gte-small')` | 把文字變成向量（取代 Cohere 的 embed） |
| 向量資料庫 | `FAISS.from_texts(texts, embeddings)` | LangChain 包好的 FAISS：一行就完成「嵌入 + 建索引」 |
| 提示詞 | `PromptTemplate.from_template(...)` | 含 `{context}`、`{question}` 的模板 |

`LlamaCpp` 的參數：

| 參數 | 意思 |
|---|---|
| `n_gpu_layers=-1` | 所有層都放到 GPU（沒有 GPU 時改成 `0`） |
| `max_tokens=500` | 最多生成 500 個 token |
| `n_ctx=2048` | 上下文視窗：提示詞 + 答案加起來不能超過 2048 個 token |
| `seed=42` | 固定隨機種子 |
| `verbose=False` | 不印出 llama.cpp 的載入訊息 |

> 這裡沒有設定 `temperature`，LangChain 的 `LlamaCpp` 預設是 0.8，所以回答帶有隨機性；`seed=42` 讓它比較可重複，但想要更穩定的輸出，通常會把 `temperature` 設成 0。

### 提示詞模板

Phi-3 的聊天格式是 `<|user|> ... <|end|> <|assistant|>`，所以模板要照這個格式寫。`{context}` 會被填入檢索到的段落，`{question}` 填入使用者的問題。這兩個名字是 `RetrievalQA` 的 `stuff` 鏈預設用的名字，不能亂改。

In [16]:
LOCAL_LLM_MODEL = "../Phi-3-mini-4k-instruct-q4.gguf"
LOCAL_EMBED_MODEL = "thenlper/gte-small"

LOCAT_PROMPT_TEMPLATE = """
<|user|>
Relevant information:
{context}                             

Provide a concise answer to the following question using the relevant information provided above:
{question}
<|end|>
<|assistant|>
"""

local_llm = LlamaCpp(
        model_path = LOCAL_LLM_MODEL,
        n_gpu_layers = -1,
        max_tokens = 500,
        n_ctx = 2048,
        seed = 42,
        verbose=False,
)

local_embed_mdoel = HuggingFaceEmbeddings(               #將文本轉換為數值表達法的嵌入模型
    model_name=LOCAL_EMBED_MODEL,
)

local_db = FAISS.from_texts(                         #建立本地端向量資料庫
    texts,
    local_embed_mdoel,
)

local_prompt = PromptTemplate.from_template(
    template=LOCAT_PROMPT_TEMPLATE,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### 把檢索與生成串成一條鏈：`RetrievalQA`

| 參數 | 意思 |
|---|---|
| `llm` | 生成模型（Phi-3） |
| `retriever=db.as_retriever(search_kwargs={"k": 6})` | 把向量資料庫變成「檢索器」，每次取最相近的 6 塊 |
| `chain_type='stuff'` | 把檢索到的**全部**文件「塞」進同一個提示詞的 `{context}` |
| `chain_type_kwargs={"prompt": ...}` | 用我們自己的提示詞模板 |
| `verbose=True` | 印出 `Entering new chain` / `Finished chain` 等執行過程 |

`rag.invoke({"query": query})` 回傳 dict：`{'query': 問題, 'result': 答案}`。答案在 `['result']`。

> **為什麼這裡也用函式？** 同樣的理由：輸入問題、輸出結果，可以反覆呼叫。

### 小模型讀資料的限制

你之前跑出的一次結果是：「最初上映賺 6.81 億，後續上映**再加** 7.747 億，總共約 14.55 億」。但原文是「**累計**含後續上映共 7.747 億」。也就是 Phi-3-mini 把「累計數」誤當成「額外增加的金額」。我沒有針對這題重現或測試過，但這是小模型讀資料時常見的失誤。RAG 讓模型看到正確資料，並不保證它讀得正確，所以答案仍要對照原文檢查。

In [19]:
def local_rag_generation(query):
    rag = RetrievalQA.from_chain_type(                #RAG流程
        llm = local_llm,
        retriever = local_db.as_retriever(search_kwargs={"k": 6}),
        chain_type = 'stuff',                         #stuff 這條鏈預設把文件填進名為 context 的變數，把問題填進 question
        chain_type_kwargs = {
            "prompt" : local_prompt
        },
        verbose = True
    )

    response = rag.invoke(
        {"query" : query}
    )

    return response

In [20]:
local_rag_resposne = local_rag_generation(query_2)
print(local_rag_resposne)



> Entering new RetrievalQA chain...

> Finished chain.
{'query': 'How much income did Interstellar generate?', 'result': ' Interstellar grossed $681 million during its initial theatrical run and an additional $774.7 million with subsequent releases, totaling approximately $1.455 billion in global revenue.'}


## 12. 用完要關閉模型：`close`

程式結束時如果沒有先關閉 Phi-3，Python 在收尾時才自動清理，可能會看到 `Exception ignored in: <function Llama.__del__ ...>` 這樣的警告（不影響結果）。解法是**自己先關**：

- LangChain 的 `LlamaCpp` 沒有 `close()`（我檢查過 `langchain-community 0.4.2` 的原始碼）。
- 真正載入模型的是它裡面的 `client`（`llama_cpp.Llama` 物件），所以要關 `local_llm.client`。
- 關完把 `local_llm` 設成 `None`；`if local_llm is not None` 的判斷，是為了「沒執行過本地 RAG」的情況不要出錯。

> 我的環境沒有安裝 `llama-cpp-python`，所以**沒有實際驗證**這一步會不會讓警告消失。

In [21]:
if local_llm is not None:
    local_llm.client.close()
    local_llm = None

## 13. 三種檢索的比較

| | 密集檢索（嵌入向量） | BM25 關鍵字檢索 | BM25 + Rerank |
|---|---|---|---|
| 依據 | 意思相近 | 詞有沒有出現 | 先比對詞，再由模型判斷相關性 |
| 擅長 | 換句話說、同義詞 | 專有名詞、型號、確切的字 | 在候選裡挑出最相關的 |
| 弱點 | 需要嵌入模型（API 或本地） | **詞不一樣就找不到** | 多一次 Rerank 呼叫 |
| 本課的證據 | — | 第 8 節實驗：`query_2` 問 `income`，文章寫的是 `grossing`，BM25 最高分的不是含 `$681` 的塊 | — |

「本課的證據」那一格是我用真實的 BM25 與這篇文章跑出來的結果（`income`、`generate` 不在文章裡，只有 `interstellar` 能比對，所以分數高的是講到 Interstellar 次數多的塊）。至於密集檢索在 `query_2` 上是否找得到含 `$681` 的塊，要靠真的呼叫 Cohere，我沒有驗證。

## 14. 重點整理

| # | 重點 |
|---|---|
| 1 | **RAG = 檢索 + 生成。** 先從自己的文件找資料，再讓 LLM 依據資料回答 |
| 2 | **切塊是檢索的基礎。** 這份程式用「3 句一塊、每次移 1 句」的滑動視窗，相鄰塊會重疊 |
| 3 | **嵌入把文字變成向量。** 文件用 `search_document`，問題用 `search_query` |
| 4 | **`IndexFlatL2` 回傳距離的平方，越小越相似。** 向量要先轉成 `float32` |
| 5 | **FAISS 找不到足夠結果時編號是 `-1`**，直接拿去索引陣列會悄悄取到最後一筆 |
| 6 | **BM25 只看詞，不懂意思。** 查詢詞不在文章裡，就拿不到分數 |
| 7 | **兩階段檢索：** 便宜的方法先粗篩，貴的精準方法再精排 |
| 8 | `np.argpartition` 只做部分排序，所以後面還要 `sorted`；`num_candidates` 不能超過塊數 |
| 9 | **Grounded generation：** 把 `documents` 和 `message` 一起交給 `chat` |
| 10 | **本地 RAG：** `LlamaCpp` + `HuggingFaceEmbeddings` + `FAISS.from_texts` + `RetrievalQA` 的 `stuff` 鏈 |
| 11 | **RAG 不保證答對。** 小模型可能讀錯資料；要看檢索到什麼、再對照答案 |
| 12 | 用完要關模型：`local_llm.client.close()` |

## 15. 程式小提醒

| # | 觀察 | 說明 |
|---|---|---|
| 1 | 變數名稱拼字 | `LOCAT_PROMPT_TEMPLATE`、`local_embed_mdoel`、`resposne` 少打或打反了字母。只要前後一致就不會出錯，但別人讀起來容易困惑 |
| 2 | 輸出標題與實際數量不符 | 程式印 `Top 3 lexical search`、`Top3 hits by rerank-API`，但實際印出的數量是 `top_k`（預設 5） |
| 3 | `Bros.` 被切成兩句 | 第 4 節的實驗證實了。因為塊會重疊，影響不大，但切句規則有這個盲點 |
| 4 | `num_candidates` 的上限 | 必須 ≤ 塊數（這篇文章是 13 塊）。再大 `np.argpartition` 會丟 `ValueError` |
| 5 | `BM25Okapi` 每次搜尋都重新建立 | 在 `keyword_and_rerank_search` 裡建立，資料多時會浪費時間；通常只建立一次 |
| 6 | 私有模組 | `from sklearn.feature_extraction import _stop_words` 的 `_stop_words` 以底線開頭，屬於內部使用；公開寫法是 `from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS`（我確認過兩者是同一個集合） |
| 7 | 取回的塊重複 | 13 塊裡最多只有 5 塊彼此不重疊，所以 `k=6` 取回的塊**一定**有重疊，`{context}` 裡有重複的句子，會佔用 `n_ctx=2048` |
| 8 | `number_of_results` 比塊數大 | 見第 6 節的實驗：FAISS 回傳 `-1`，`dense_search` 會悄悄把最後一塊當作結果 |

## 16. 動手練習

1. 把 `# print(...)` 前面的 `#` 拿掉，逐步觀察 `texts`、`texts_embeds.shape`、`similar_item_ids` 的內容。
2. 把 `window = 3` 改成 `1` 和 `5`，比較塊數，以及 `dense_search` 的結果有什麼不同。
3. 用 `dense_search(query_2)` 看前幾名有沒有含 `$681` 的塊，並與第 8 節 BM25 的結果比較。
4. 把 `dense_search` 的 `number_of_results` 改成 20，**先預測**會發生什麼事，再執行驗證。
5. 把 `k=6` 改成 `3`，比較 Phi-3 的答案和 `{context}` 的長度。
6. 修正 `Bros.` 的切句問題。（提示：在 `re.split` 的規則前面加上 `(?<!Bros\.)`，表示「前面不是 `Bros.`」才切。）
7. 在提示詞模板最後加一句 `Do not add numbers together unless asked.`，看 Phi-3 的答案有沒有改善。（**我沒有測試過這個做法有沒有用**）
8. 把 `keyword_and_rerank_search` 裡的 `BM25Okapi(tokenized_corpus)` 移到函式外面，只建立一次。

## 17. 延伸方向

- 嵌入模型與索引種類：`IndexFlatIP`（內積）、近似搜尋的索引（資料量大時）
- **混合檢索（hybrid search）**：同時用密集檢索與 BM25，再合併結果
- 用 LangChain 的 LCEL 自己組 `retriever | prompt | llm`，取代舊式的 `RetrievalQA`
- Cohere 另有較新的客戶端 `cohere.ClientV2`（本程式用的是 `cohere.Client`）；新舊介面的差異我沒有查證，使用前請看官方文件
- 評估 RAG：檢索是否找到正確的塊？答案是否忠於資料？